# 02 — Data Cleaning & Panel Construction

**Author:** Ricardo Sanchez  

**Project:** Mortgage Rate Lock-In Effect on U.S. Housing Supply

**Date:** March 2026  

**Data Source:** 9 raw CSV files in `data/raw/`

---

## Objective

Transform 9 raw CSV files into a single analysis-ready quarterly panel. Each row represents one state-quarter observation with the rate gap, controls, and dependent variables aligned and ready for regression.

## Inputs / Outputs

**Input:** `data/raw/` (9 CSVs validated in `01_pre_merge_eda.ipynb`)  
**Output:** `data/panel/quarterly_panel.csv` (2,601 rows × 28 columns)

### Pipeline Overview
1. Load all raw files
2. Reshape & standardize each dataset (name mapping, date conversion, frequency alignment)
3. Aggregate monthly data to quarterly
4. Compute derived variables (rate gap, YoY growth, share locked-in, lagged migration)
5. Merge everything into the final panel
6. Post-merge validation

### Aggregation Rules (Monthly → Quarterly)
| Data Type | Rule | Example |
|-----------|------|---------|
| Counts | **SUM** within quarter | New listings, permits |
| Rates / Indices | **AVERAGE** within quarter | Unemployment, HPI, mortgage rate |
| Point-in-time | **End-of-quarter** (last month) | Active inventory |
| Annual | Assign to 4 quarters, **lag 1 year** | Net migration |

---

## 1. Setup & Configuration

In [1]:
import os, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

RAW = os.path.join("..", "data", "raw")
PANEL_DIR = os.path.join("..", "data", "panel")
os.makedirs(PANEL_DIR, exist_ok=True)

ALL_ST = sorted(["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
    "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
    "VT","VA","WA","WV","WI","WY"])

STATE_NAME_TO_ABBR = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","District of Columbia":"DC",
    "Florida":"FL","Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL",
    "Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY","Louisiana":"LA",
    "Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI","Minnesota":"MN",
    "Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV",
    "New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR",
    "Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD",
    "Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA",
    "Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY"
}

print("Setup complete")

Setup complete


---

# Part 1 — Load & Reshape Each Dataset

Each dataset is loaded, cleaned, and converted to a common long format: **`state` (2-letter) × `date` (datetime) × variables**. Quarterly aggregation happens in Part 2.

Issues identified in NB01 are resolved here: FHFA duplicate rows (MARKET filter), Realtor.com column conflict (`state` vs `state_id`), unemployment Oct 2025 nulls (linear interpolation), and Zillow full-name-to-abbreviation mapping.

### 1.1 FHFA Outstanding Mortgage Statistics

The most critical dataset. Contains `AVE_INTRATE` (average outstanding mortgage rate by state/quarter) and rate distribution buckets. This builds the lock-in rate gap variable.

*NB01 finding: FHFA file has multiple MARKET values creating duplicates per state-quarter. Filter to "All Mortgages" or aggregate with groupby.*

In [2]:
# Load FHFA 
fhfa_raw = pd.read_csv(os.path.join(RAW, "nmdb-outstanding-mortgage-statistics-states-quarterly.csv"))
print(f"Raw shape: {fhfa_raw.shape[0]:,} rows")

# Filter to state-level only
fhfa = fhfa_raw[fhfa_raw["GEOLEVEL"] == "State"].copy()
print(f"State-level: {len(fhfa):,} rows")

# Check MARKET column — filter to 'All Mortgages' if multiple exist
if "MARKET" in fhfa.columns:
    markets = fhfa["MARKET"].unique()
    print(f"MARKET values: {markets}")
    # Keep only the broadest market category
    all_mort = [m for m in markets if "All" in str(m)]
    if all_mort:
        fhfa = fhfa[fhfa["MARKET"] == all_mort[0]]
        print(f"Filtered to '{all_mort[0]}': {len(fhfa):,} rows")

# Extract the series we need
keep_series = ["AVE_INTRATE", "PCT_INTRATE_LT_3", "PCT_INTRATE_3_4", 
               "PCT_INTRATE_4_5", "PCT_INTRATE_5_6", "PCT_INTRATE_GE_6", "TOT_LOANS"]

fhfa = fhfa[fhfa["SERIESID"].isin(keep_series)].copy()
print(f"After series filter: {len(fhfa):,} rows")

# Pivot: one row per state-quarter, columns = series
fhfa_pivot = fhfa.pivot_table(
    index=["GEOID", "PERIOD"], columns="SERIESID", values="VALUE1", aggfunc="mean"
).reset_index()
fhfa_pivot.columns.name = None

# Rename columns for clarity
fhfa_pivot = fhfa_pivot.rename(columns={
    "GEOID": "state", "PERIOD": "quarter_str",
    "AVE_INTRATE": "ave_intrate",
    "PCT_INTRATE_LT_3": "pct_lt3", "PCT_INTRATE_3_4": "pct_3_4",
    "PCT_INTRATE_4_5": "pct_4_5", "PCT_INTRATE_5_6": "pct_5_6",
    "PCT_INTRATE_GE_6": "pct_ge6", "TOT_LOANS": "tot_loans"
})

# Create proper date column from quarter string (e.g., "2013Q1" -> 2013-01-01)
fhfa_pivot["date"] = pd.PeriodIndex(fhfa_pivot["quarter_str"], freq="Q").to_timestamp()

print(f"\nFHFA pivoted: {len(fhfa_pivot)} rows × {fhfa_pivot.shape[1]} cols")
print(f"States: {fhfa_pivot['state'].nunique()}")
print(f"Quarters: {fhfa_pivot['quarter_str'].min()} to {fhfa_pivot['quarter_str'].max()}")
fhfa_pivot.head(3)

Raw shape: 384,948 rows
State-level: 384,948 rows
MARKET values: ['All Mortgages' 'Enterprise Acquisitions' 'Government / Non-Conventional'
 'Other Conventional Market']
Filtered to 'All Mortgages': 96,237 rows
After series filter: 18,207 rows

FHFA pivoted: 2601 rows × 10 cols
States: 51
Quarters: 2013Q1 to 2025Q3


,state,quarter_str,ave_intrate,pct_3_4,pct_4_5,pct_5_6,pct_ge6,pct_lt3,tot_loans,date
0,AK,2013Q1,5.0,23.3,29.1,21.9,22.1,3.5,118.0,2013-01-01
1,AK,2013Q2,4.9,26.4,28.7,19.9,20.6,4.4,117.0,2013-04-01
2,AK,2013Q3,4.8,27.2,29.8,19.0,19.4,4.6,117.0,2013-07-01


### 1.2 Building Permits

Wide-to-long melt. Monthly frequency — will be summed to quarterly in Part 2.

In [3]:
permits_wide = pd.read_csv(os.path.join(RAW, "state_permits.csv"), index_col=0, parse_dates=True)

# Melt wide to long
permits = permits_wide.reset_index().melt(id_vars="date", var_name="state", value_name="permits")
permits["date"] = pd.to_datetime(permits["date"])

print(f"Permits: {len(permits):,} rows (monthly, long format)")
print(f"Nulls: {permits['permits'].isnull().sum()}")
permits.head(3)

Permits: 15,912 rows (monthly, long format)
Nulls: 1


,date,state,permits
0,2000-01-01,AL,1495.819938
1,2000-02-01,AL,1523.061581
2,2000-03-01,AL,1575.742515


### 1.3 FHFA House Price Index

Wide-to-long melt. HPI YoY is computed **before** merging onto the FHFA backbone so that `pct_change(4)` can reference pre-2013 history.

In [4]:
hpi_wide = pd.read_csv(os.path.join(RAW, "state_hpi.csv"), index_col=0, parse_dates=True)

# Melt wide -> long
hpi = hpi_wide.reset_index().melt(id_vars="date", var_name="state", value_name="hpi")
hpi["date"] = pd.to_datetime(hpi["date"])

print(f"HPI: {len(hpi):,} rows")
hpi.head(6)

HPI: 15,810 rows


,date,state,hpi
0,2000-01-01,AL,207.47
1,2000-02-01,AL,207.47
2,2000-03-01,AL,207.47
3,2000-04-01,AL,209.23
4,2000-05-01,AL,209.23
5,2000-06-01,AL,209.23


### 1.4 State Unemployment

Wide-to-long melt. Monthly frequency — will be averaged to quarterly in Part 2.

In [5]:
unemp_wide = pd.read_csv(os.path.join(RAW, "state_unemployment.csv"), index_col=0, parse_dates=True)
unemp = unemp_wide.reset_index().melt(id_vars="date", var_name="state", value_name="unemp_rate")
unemp["date"] = pd.to_datetime(unemp["date"])

print(f"Unemployment: {len(unemp):,} rows")
print(f"Nulls: {unemp['unemp_rate'].isnull().sum()}")
unemp.head(3)

Unemployment: 15,912 rows
Nulls: 51


,date,state,unemp_rate
0,2000-01-01,AL,4.7
1,2000-02-01,AL,4.7
2,2000-03-01,AL,4.7


In [6]:
# ── Impute Oct 2025 unemployment nulls: average of Sep and Nov ──
nulls_before = unemp["unemp_rate"].isnull().sum()

unemp["unemp_rate"] = (
    unemp.groupby("state")["unemp_rate"]
    .transform(lambda x: x.interpolate(method="linear", limit=1))
)

nulls_after = unemp["unemp_rate"].isnull().sum()
print(f"Unemployment nulls: {nulls_before} → {nulls_after}")
print(f"Method: linear interpolation (average of adjacent months)")

Unemployment nulls: 51 → 0
Method: linear interpolation (average of adjacent months)


### 1.5 National Macro Series

Only `mortgage_rate_30yr` enters the panel directly (for rate gap computation). Other national series (fed funds, lumber PPI, construction PPI, CPI shelter) are absorbed by time fixed effects.

In [7]:
natl = pd.read_csv(os.path.join(RAW, "national_series.csv"), index_col=0, parse_dates=True)
natl = natl[["mortgage_rate_30yr"]].copy()  # only need mortgage rate for rate gap

print(f"National mortgage rate: {len(natl)} months")
print(f"Range: {natl['mortgage_rate_30yr'].min():.2f}% to {natl['mortgage_rate_30yr'].max():.2f}%")
natl.head(3)

National mortgage rate: 312 months
Range: 2.68% to 8.52%


,mortgage_rate_30yr
date,
2000-01-01,8.210
2000-02-01,8.325
2000-03-01,8.240


### 1.6 Zillow ZHVI

Uses full state names (`RegionName`) which must be mapped to 2-letter abbreviations. Alternative price measure for robustness — not used in primary specification.

In [8]:
zillow_raw = pd.read_csv(os.path.join(RAW, "State_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))
date_cols = [c for c in zillow_raw.columns if c[:4].isdigit()]

# Map full state names to abbreviations
zillow_raw["state"] = zillow_raw["RegionName"].map(STATE_NAME_TO_ABBR)
unmapped = zillow_raw[zillow_raw["state"].isnull()]["RegionName"].tolist()
if unmapped:
    print(f"WARNING: Unmapped states: {unmapped}")

# Melt wide to long
zillow = zillow_raw[["state"] + date_cols].melt(id_vars="state", var_name="date_str", value_name="zhvi")
zillow["date"] = pd.to_datetime(zillow["date_str"])
zillow = zillow.drop(columns="date_str")

print(f"Zillow: {len(zillow):,} rows")
print(f"Nulls: {zillow['zhvi'].isnull().sum()}")
zillow.head(5)

Zillow: 16,014 rows
Nulls: 228


,state,zhvi,date
0,CA,187986.834929,2000-01-31
1,TX,112895.976760,2000-01-31
2,FL,107549.759105,2000-01-31
3,NY,150045.977338,2000-01-31
4,PA,98231.424861,2000-01-31


### 1.7 Realtor.com Inventory

*Issue from NB01: file has both `state` (full name) and `state_id` (abbreviation). Drop `state`, rename `state_id` to `state` to avoid column conflict in the merge.*

In [9]:
rdc = pd.read_csv(os.path.join(RAW, "RDC_Inventory_Core_Metrics_State_History.csv"))

# Convert integer date to datetime
rdc["date"] = pd.to_datetime(rdc["month_date_yyyymm"].astype(str), format="%Y%m")

# Drop full state name column, keep abbreviation (state_id)
rdc = rdc.drop(columns=["state"]).rename(columns={"state_id": "state"})

# Keep only the columns we need
rdc_cols = ["state", "date", "new_listing_count", "active_listing_count", 
            "median_days_on_market", "median_listing_price"]
rdc = rdc[rdc_cols].copy()

print(f"Realtor.com: {len(rdc):,} rows")
print(f"Date range: {rdc['date'].min().date()} to {rdc['date'].max().date()}")
rdc.head(3)

Realtor.com: 5,916 rows
Date range: 2016-07-01 to 2026-02-01


,state,date,new_listing_count,active_listing_count,median_days_on_market,median_listing_price
0,GA,2026-02-01,14688,42664,66,383979
1,MO,2026-02-01,7558,16213,70,295000
2,TX,2026-02-01,38330,117736,70,349900


### 1.8 Census Net Domestic Migration

Two vintage files combined (2010–2020 and 2020–2025). The 2020s vintage takes precedence for overlapping years. Migration is assigned to 4 quarters per year and **lagged by 1 year** (migration from year *t-1* used for year *t* quarters).

*Design decision: net domestic migration chosen over population growth because migration is a component of pop growth — including both is redundant.

In [10]:
# Load both vintages
c10 = pd.read_csv(os.path.join(RAW, "nst-est2020-alldata.csv"))
c20 = pd.read_csv(os.path.join(RAW, "NST-EST2025-ALLDATA.csv"))

# Filter to states, exclude Puerto Rico
c10_st = c10[(c10["SUMLEV"]==40) & (c10["NAME"]!="Puerto Rico")]
c20_st = c20[(c20["SUMLEV"]==40) & (c20["NAME"]!="Puerto Rico")]

# Extract RDOMESTICMIG (rate per 1,000) into long format
rows = []
for _, row in c10_st.iterrows():
    st = STATE_NAME_TO_ABBR.get(row["NAME"])
    if not st: continue
    for col in [c for c in c10_st.columns if c.startswith("RDOMESTICMIG")]:
        year = int(col.replace("RDOMESTICMIG", ""))
        rows.append({"state": st, "year": year, "rdomesticmig": row[col]})

for _, row in c20_st.iterrows():
    st = STATE_NAME_TO_ABBR.get(row["NAME"])
    if not st: continue
    for col in [c for c in c20_st.columns if c.startswith("RDOMESTICMIG")]:
        year = int(col.replace("RDOMESTICMIG", ""))
        rows.append({"state": st, "year": year, "rdomesticmig": row[col]})

# Keep most recent vintage for overlapping years (2020s supersedes 2010s)
mig = pd.DataFrame(rows).drop_duplicates(subset=["state","year"], keep="last").sort_values(["state","year"]).reset_index(drop=True)

print(f"Migration: {len(mig)} rows ({mig['state'].nunique()} states, {mig['year'].min()}-{mig['year'].max()})")
print(f"Rate range: {mig['rdomesticmig'].min():.2f} to {mig['rdomesticmig'].max():.2f} per 1,000")
mig.head()

Migration: 765 rows (51 states, 2011-2025)
Rate range: -16.33 to 27.89 per 1,000


,state,year,rdomesticmig
0,AK,2011,-1.236484
1,AK,2012,-1.822237
2,AK,2013,-4.769701
3,AK,2014,-12.999245
4,AK,2015,-11.110772


---

# Part 2 — Aggregate to Quarterly

All monthly series are collapsed to quarterly to match the FHFA's native frequency. This eliminates forward-fill staleness and ensures each observation represents a true quarterly value.

In [11]:
def assign_quarter(df, date_col="date"):
    """Add quarter columns to a monthly dataframe."""
    df = df.copy()
    df["year"] = df[date_col].dt.year
    df["quarter"] = df[date_col].dt.quarter
    df["yq"] = df["year"].astype(str) + "Q" + df["quarter"].astype(str)
    # Create quarter start date for merging
    df["qdate"] = pd.PeriodIndex(df["yq"], freq="Q").to_timestamp()
    return df

# ── 2a. Permits: SUM within quarter ──
permits_q = assign_quarter(permits)
permits_q = permits_q.groupby(["state","qdate"]).agg(permits=("permits","sum")).reset_index()
print(f"Permits quarterly: {len(permits_q):,} rows")

# ── 2b. HPI: AVERAGE within quarter, compute YoY BEFORE merge ──
hpi_q = assign_quarter(hpi)
hpi_q = hpi_q.groupby(["state","qdate"]).agg(hpi=("hpi","mean")).reset_index()

# Compute YoY here while full history (back to 2000) is available
hpi_q = hpi_q.sort_values(["state","qdate"])
hpi_q["hpi_yoy"] = hpi_q.groupby("state")["hpi"].pct_change(4) * 100

print(f"HPI quarterly: {len(hpi_q):,} rows")
print(f"HPI YoY nulls: {hpi_q['hpi_yoy'].isnull().sum()} (only 2000 Q1-Q4, outside panel)")

# ── 2c. Unemployment: AVERAGE within quarter ──
unemp_q = assign_quarter(unemp)
unemp_q = unemp_q.groupby(["state","qdate"]).agg(unemp_rate=("unemp_rate","mean")).reset_index()
print(f"Unemployment quarterly: {len(unemp_q):,} rows")

# ── 2d. Mortgage rate: AVERAGE within quarter ──
natl_q = natl.copy()
natl_q = natl_q.reset_index()
natl_q = assign_quarter(natl_q, "date")
natl_q = natl_q.groupby("qdate").agg(mortgage_rate_30yr=("mortgage_rate_30yr","mean")).reset_index()
print(f"Mortgage rate quarterly: {len(natl_q)} quarters")

# ── 2e. Zillow ZHVI: AVERAGE within quarter ──
zillow_q = assign_quarter(zillow)
zillow_q = zillow_q.groupby(["state","qdate"]).agg(zhvi=("zhvi","mean")).reset_index()

zillow_q = zillow_q.sort_values(["state","qdate"])
zillow_q["zhvi_yoy"] = zillow_q.groupby("state")["zhvi"].pct_change(4) * 100

print(f"Zillow quarterly: {len(zillow_q):,} rows")

# ── 2f. Realtor.com: SUM for counts, AVERAGE for rates ──
rdc_q = assign_quarter(rdc)
rdc_q = rdc_q.groupby(["state","qdate"]).agg(
    new_listing_count=("new_listing_count", "sum"),
    active_listing_count=("active_listing_count", "last"),  # end-of-quarter snapshot
    median_days_on_market=("median_days_on_market", "mean"),
    median_listing_price=("median_listing_price", "mean"),
).reset_index()
print(f"Realtor.com quarterly: {len(rdc_q):,} rows")

# ── 2g. Migration: assign annual to quarters, lag by 1 year ──
# Each Census year (July Y-1 to June Y) maps to calendar year Y
# Lag by 1 year: migration from year Y-1 used for year Y quarters
mig_lagged = mig.copy()
mig_lagged["year"] = mig_lagged["year"] + 1  # lag by 1 year
mig_lagged = mig_lagged.rename(columns={"rdomesticmig": "net_migration_lag1"})

# Expand to 4 quarters per year
mig_quarterly = []
for _, row in mig_lagged.iterrows():
    for q in range(1, 5):
        qdate = pd.Timestamp(f"{int(row['year'])}-{(q-1)*3+1:02d}-01")
        mig_quarterly.append({"state": row["state"], "qdate": qdate, "net_migration_lag1": row["net_migration_lag1"]})

mig_q = pd.DataFrame(mig_quarterly)
print(f"Migration quarterly (lagged): {len(mig_q):,} rows")

Permits quarterly: 5,304 rows
HPI quarterly: 5,304 rows
HPI YoY nulls: 204 (only 2000 Q1-Q4, outside panel)
Unemployment quarterly: 5,304 rows
Mortgage rate quarterly: 104 quarters
Zillow quarterly: 5,355 rows
Realtor.com quarterly: 1,989 rows
Migration quarterly (lagged): 3,060 rows


---

# Part 3 — Compute Derived Variables

Compute key analytical variables on the FHFA data before merging: `share_locked_in` (sum of rate buckets below 5%)..

In [12]:
# Rename FHFA date column for merge compatibility
fhfa_panel = fhfa_pivot.rename(columns={"date": "qdate"}).copy()

# ShareLockedIn: % of mortgages with rates below 5%
if all(c in fhfa_panel.columns for c in ["pct_lt3","pct_3_4","pct_4_5"]):
    fhfa_panel["share_locked_in"] = fhfa_panel["pct_lt3"] + fhfa_panel["pct_3_4"] + fhfa_panel["pct_4_5"]
    print(f"ShareLockedIn computed: mean={fhfa_panel['share_locked_in'].mean():.1f}%")
else:
    print("WARNING: Rate bucket columns not found — check FHFA series filter")

print(f"\nFHFA panel ready: {len(fhfa_panel)} rows, {fhfa_panel['state'].nunique()} states")
print(f"Columns: {list(fhfa_panel.columns)}")

ShareLockedIn computed: mean=71.7%

FHFA panel ready: 2601 rows, 51 states
Columns: ['state', 'quarter_str', 'ave_intrate', 'pct_3_4', 'pct_4_5', 'pct_5_6', 'pct_ge6', 'pct_lt3', 'tot_loans', 'qdate', 'share_locked_in']


---

# Part 4 — Merge Into Final Panel

All datasets are merged on `(state, qdate)` using left joins onto the FHFA backbone. The FHFA data determines the panel's time boundaries (2013 Q1 – 2025 Q3) since it's the binding constraint.

After each merge, row counts and null counts are reported to track exactly where data drops occur.

In [13]:
# Start with FHFA as the backbone
panel = fhfa_panel[["state", "qdate", "quarter_str", "ave_intrate", 
                      "pct_lt3", "pct_3_4", "pct_4_5", "pct_5_6", "pct_ge6",
                      "tot_loans", "share_locked_in"]].copy()

print(f"Starting panel: {len(panel)} rows (FHFA backbone)")

# 4a. Merge mortgage rate (national, by qdate only)
panel = panel.merge(natl_q, on="qdate", how="left")
print(f"After mortgage rate: {len(panel)} rows, nulls={panel['mortgage_rate_30yr'].isnull().sum()}")

# 4b. Compute RateGap
panel["rate_gap"] = panel["mortgage_rate_30yr"] - panel["ave_intrate"]
print(f"Rate gap: mean={panel['rate_gap'].mean():.2f}, min={panel['rate_gap'].min():.2f}, max={panel['rate_gap'].max():.2f}")

# 4c. Merge permits
panel = panel.merge(permits_q, on=["state","qdate"], how="left")
print(f"After permits: {len(panel)} rows, nulls={panel['permits'].isnull().sum()}")

# 4d. Merge HPI
panel = panel.merge(hpi_q[["state", "qdate", "hpi", "hpi_yoy"]], on=["state", "qdate"], how="left")
print(f"After HPI: {len(panel)} rows, hpi_yoy nulls={panel['hpi_yoy'].isnull().sum()}")

# 4e. Merge unemployment
panel = panel.merge(unemp_q, on=["state","qdate"], how="left")
print(f"After unemployment: {len(panel)} rows, nulls={panel['unemp_rate'].isnull().sum()}")

# 4f. Merge Zillow ZHVI
panel = panel.merge(zillow_q[["state", "qdate", "zhvi", "zhvi_yoy"]], on=["state","qdate"], how="left")
print(f"After Zillow: {len(panel)} rows, nulls={panel['zhvi'].isnull().sum()}")

# 4g. Merge Realtor.com
panel = panel.merge(rdc_q, on=["state","qdate"], how="left")
print(f"After Realtor.com: {len(panel)} rows, nulls in new_listing_count={panel['new_listing_count'].isnull().sum()}")

# 4h. Merge migration
panel = panel.merge(mig_q, on=["state","qdate"], how="left")
print(f"After migration: {len(panel)} rows, nulls={panel['net_migration_lag1'].isnull().sum()}")

# 4i. COVID dummy
panel["covid_dummy"] = ((panel["qdate"] >= "2020-04-01") & (panel["qdate"] <= "2020-09-30")).astype(int)
print(f"COVID dummy: {panel['covid_dummy'].sum()} quarter-states flagged")

# 4j. Log-transform dependent variables
panel["ln_permits"] = np.log1p(panel["permits"])
panel["ln_new_listings"] = np.log1p(panel["new_listing_count"])

# 4k. Post-2022 dummy (for diff-in-diff)
panel["post_2022"] = (panel["qdate"] >= "2022-04-01").astype(int)

print(f"\n{'='*60}")
print(f"FINAL PANEL: {len(panel):,} rows × {panel.shape[1]} columns")
print(f"States: {panel['state'].nunique()}")
print(f"Quarters: {panel['qdate'].min().date()} to {panel['qdate'].max().date()}")
print(f"{'='*60}")

Starting panel: 2601 rows (FHFA backbone)
After mortgage rate: 2601 rows, nulls=0
Rate gap: mean=0.08, min=-2.50, max=3.49
After permits: 2601 rows, nulls=0
After HPI: 2601 rows, hpi_yoy nulls=0
After unemployment: 2601 rows, nulls=0
After Zillow: 2601 rows, nulls=0
After Realtor.com: 2601 rows, nulls in new_listing_count=714
After migration: 2601 rows, nulls=0
COVID dummy: 102 quarter-states flagged

FINAL PANEL: 2,601 rows × 28 columns
States: 51
Quarters: 2013-01-01 to 2025-07-01


---

# Part 5 — Post-Merge Validation

Verify panel balance, check null patterns, compute summary statistics, and confirm key variable ranges are sensible before passing to the EDA and regression notebooks.

In [14]:
print("COLUMN INVENTORY:")
print(f"{'Column':<25s} {'Dtype':<12s} {'Nulls':>6s} {'Null%':>7s} {'Mean':>12s}")
print("-" * 65)
for col in panel.columns:
    n = panel[col].isnull().sum()
    pct = n / len(panel) * 100
    if panel[col].dtype in ["float64","int64"]:
        mn = f"{panel[col].mean():.2f}"
    else:
        mn = "—"
    print(f"  {col:<23s} {str(panel[col].dtype):<12s} {n:>6,} {pct:>6.1f}% {mn:>12s}")

COLUMN INVENTORY:
Column                    Dtype         Nulls   Null%         Mean
-----------------------------------------------------------------
  state                   object            0    0.0%            —
  qdate                   datetime64[ns]      0    0.0%            —
  quarter_str             object            0    0.0%            —
  ave_intrate             float64           0    0.0%         4.47
  pct_lt3                 float64           0    0.0%        10.71
  pct_3_4                 float64           0    0.0%        32.79
  pct_4_5                 float64           0    0.0%        28.16
  pct_5_6                 float64           0    0.0%        12.78
  pct_ge6                 float64           0    0.0%        15.56
  tot_loans               float64           0    0.0%       991.47
  share_locked_in         float64           0    0.0%        71.66
  mortgage_rate_30yr      float64           0    0.0%         4.55
  rate_gap                float64          

In [15]:
# Check: every state should have the same number of quarters
state_counts = panel.groupby("state")["qdate"].count()
print(f"Quarters per state: min={state_counts.min()}, max={state_counts.max()}, mode={state_counts.mode().values[0]}")
if state_counts.min() != state_counts.max():
    unbalanced = state_counts[state_counts != state_counts.mode().values[0]]
    print(f"Unbalanced states: {dict(unbalanced)}")
else:
    print("\u2713 Panel is balanced")

# Check: rate gap should be negative before 2022 (rates were lower than outstanding)
# and positive after (rates higher than outstanding)
pre = panel[panel["qdate"] < "2022-01-01"]["rate_gap"].mean()
post = panel[panel["qdate"] >= "2022-04-01"]["rate_gap"].mean()
print(f"\nRate gap sanity check:")
print(f"  Pre-2022 mean: {pre:.2f}  (should be near 0 or slightly negative)")
print(f"  Post-2022 mean: {post:.2f}  (should be positive = lock-in present)")

# Check: new listings should be null before 2016 Q3 (Realtor.com starts then)
pre_rdc = panel[panel["qdate"] < "2016-07-01"]["new_listing_count"].notna().sum()
print(f"\nNew listing values before 2016 Q3: {pre_rdc} (should be 0)")

Quarters per state: min=51, max=51, mode=51
✓ Panel is balanced

Rate gap sanity check:
  Pre-2022 mean: -0.83  (should be near 0 or slightly negative)
  Post-2022 mean: 2.45  (should be positive = lock-in present)

New listing values before 2016 Q3: 0 (should be 0)


In [16]:
# Summary statistics for key variables
key_vars = ["rate_gap", "share_locked_in", "ave_intrate", "mortgage_rate_30yr",
            "permits", "ln_permits", "new_listing_count", "ln_new_listings",
            "hpi_yoy", "unemp_rate", "median_days_on_market", "net_migration_lag1"]
existing = [v for v in key_vars if v in panel.columns]

print("KEY VARIABLE SUMMARY STATISTICS:")
display(panel[existing].describe().round(2).T)

KEY VARIABLE SUMMARY STATISTICS:


,count,mean,std,min,25%,50%,75%,max
rate_gap,2601.0,0.08,1.54,-2.50,-1.03,-0.62,1.64,3.49
share_locked_in,2601.0,71.66,11.37,31.60,65.10,73.70,80.30,93.50
ave_intrate,2601.0,4.47,0.46,3.40,4.10,4.40,4.80,6.00
mortgage_rate_30yr,2601.0,4.55,1.33,2.76,3.68,4.01,5.58,7.29
permits,2601.0,6541.50,9275.04,135.50,1523.23,3883.05,7095.78,70522.20
ln_permits,2601.0,8.14,1.16,4.92,7.33,8.26,8.87,11.16
new_listing_count,1887.0,24588.47,27237.63,1250.00,6311.00,16236.00,32071.00,154364.00
ln_new_listings,1887.0,9.59,1.06,7.13,8.75,9.70,10.38,11.95
hpi_yoy,2601.0,6.24,4.94,-9.61,3.28,5.04,7.42,32.15
unemp_rate,2601.0,4.64,1.92,1.70,3.33,4.20,5.43,24.70


---

# Part 6 — Define Model-Specific Subsets & Export

Save the full panel plus three pre-filtered subsets for direct consumption by the regression notebooks.

In [17]:
# Model A: New listings panel (2016 Q3 - 2025 Q3)
model_a = panel[panel["new_listing_count"].notna()].copy()
print(f"Model A (new listings): {len(model_a):,} rows, {model_a['state'].nunique()} states, "
      f"{model_a['qdate'].min().date()} to {model_a['qdate'].max().date()}")

# Model B: Permits panel (2013 Q1 - 2025 Q3) — full FHFA range
model_b = panel[panel["permits"].notna()].copy()
print(f"Model B (permits):      {len(model_b):,} rows, {model_b['state'].nunique()} states, "
      f"{model_b['qdate'].min().date()} to {model_b['qdate'].max().date()}")

# Model R5: Post-pandemic stress test (2022 Q2 - 2025 Q3)
model_r5 = panel[(panel["qdate"] >= "2022-04-01") & (panel["new_listing_count"].notna())].copy()
print(f"Model R5 (post-2022):   {len(model_r5):,} rows, {model_r5['state'].nunique()} states, "
      f"{model_r5['qdate'].min().date()} to {model_r5['qdate'].max().date()}")

Model A (new listings): 1,887 rows, 51 states, 2016-07-01 to 2025-07-01
Model B (permits):      2,601 rows, 51 states, 2013-01-01 to 2025-07-01
Model R5 (post-2022):   714 rows, 51 states, 2022-04-01 to 2025-07-01


In [18]:
# Save the full panel
out_path = os.path.join(PANEL_DIR, "quarterly_panel.csv")
panel.to_csv(out_path, index=False)
print(f"Panel saved: {out_path}")
print(f"  {len(panel):,} rows × {panel.shape[1]} columns")
print(f"  File size: {os.path.getsize(out_path)/1024:.0f} KB")

# Also save model-specific subsets
model_a.to_csv(os.path.join(PANEL_DIR, "model_a_listings.csv"), index=False)
model_b.to_csv(os.path.join(PANEL_DIR, "model_b_permits.csv"), index=False)
model_r5.to_csv(os.path.join(PANEL_DIR, "model_r5_post2022.csv"), index=False)
print(f"\nModel subsets saved to {PANEL_DIR}/")

print("""
\u2713 Panel construction complete.

NEXT STEPS:
  1. Run post-merge EDA (correlations, scatter plots, distributional checks)
  2. Load model_a_listings.csv for regression notebook
  3. Import quarterly_panel.csv into Power BI for dashboard
""")

Panel saved: ..\data\panel\quarterly_panel.csv
  2,601 rows × 28 columns
  File size: 703 KB

Model subsets saved to ..\data\panel/

✓ Panel construction complete.

NEXT STEPS:
  1. Run post-merge EDA (correlations, scatter plots, distributional checks)
  2. Load model_a_listings.csv for regression notebook
  3. Import quarterly_panel.csv into Power BI for dashboard



---

## Column Reference

Complete data dictionary for `quarterly_panel.csv`.

| Column | Description | Source |
|--------|-------------|-------|
| `state` | 2-letter state abbreviation | All |
| `qdate` | Quarter start date (datetime) | Constructed |
| `quarter_str` | Quarter label (e.g., "2023Q2") | FHFA |
| `ave_intrate` | Avg outstanding mortgage rate (%) | FHFA |
| `pct_lt3` through `pct_ge6` | Rate distribution buckets (%) | FHFA |
| `share_locked_in` | % mortgages below 5% rate | Derived (FHFA) |
| `tot_loans` | Total outstanding loans (1000s) | FHFA |
| `mortgage_rate_30yr` | Quarterly avg 30-yr rate (%) | FRED |
| `rate_gap` | Current rate minus outstanding rate (pp) | Derived |
| `permits` | Quarterly building permits (sum) | Census/FRED |
| `ln_permits` | ln(1 + permits) | Derived |
| `hpi` | FHFA House Price Index | FHFA/FRED |
| `hpi_yoy` | HPI year-over-year growth (%) | Derived |
| `unemp_rate` | Quarterly avg unemployment rate (%) | BLS/FRED |
| `zhvi` | Zillow Home Value Index ($) | Zillow |
| `zhvi_yoy` | ZHVI year-over-year growth (%) | Derived |
| `new_listing_count` | Quarterly new listings (sum) | Realtor.com |
| `active_listing_count` | End-of-quarter active inventory | Realtor.com |
| `median_days_on_market` | Quarterly avg days on market | Realtor.com |
| `median_listing_price` | Quarterly avg listing price ($) | Realtor.com |
| `ln_new_listings` | ln(1 + new listings) | Derived |
| `net_migration_lag1` | Lagged net domestic migration (per 1,000) | Census |
| `covid_dummy` | 1 if 2020 Q2-Q3, else 0 | Constructed |
| `post_2022` | 1 if 2022 Q2+, else 0 | Constructed |